# Can we predict whether a student passes the closing quiz?
Target: `passed_quiz`. Following `checklist.md`.

**Data facts that shape the design**
- There is no date/time column and row order is not a timeline, so a 'latest rows' split is not possible.
- A student appears in several rows (about 7 sessions each), and `enrol_year` and `school_code` are constant within a student. So every split (test set and CV folds) holds **whole students**.
- `student_key` is an identifier and is dropped.
- `session_minutes` and `hints_used` are recorded during the session, before the closing quiz result, so they are available when we predict the quiz outcome. `prior_score` is a placement-test score stored on the session record. The prediction moment is therefore 'quiz about to start'. I also include a candidate that uses only `prior_score`, which is known before the session starts.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, log_loss, balanced_accuracy_score, accuracy_score
df = pd.read_csv('data.csv')
print(df.shape, df.isna().sum().sum(), 'missing;', df.student_key.nunique(), 'students; pass rate', df.passed_quiz.mean().round(3))

## 1. Hold out a test set (whole students) before any modelling

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
tr_idx, te_idx = next(gss.split(df, groups=df.student_key))
train, test = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)
assert not set(train.student_key) & set(test.student_key)
print(len(train), len(test), train.student_key.nunique(), test.student_key.nunique())
# test is not touched again until step 6

## 4. Candidate models, named before any is scored
Metric for model choice: ROC AUC (threshold-free); log loss reported too.
1. `baseline_prior` - predicts the training pass rate for everyone (trivial baseline).
2. `logit_prior` - logistic regression on `prior_score` only (pre-session information).
3. `logit_session` - logistic regression on `prior_score`, `session_minutes`, `hints_used`.
4. `logit_all` - as 3 plus `enrol_year` and one-hot `school_code`.
5. `hgb_session` - gradient boosting on `prior_score`, `session_minutes`, `hints_used`.
6. `hgb_all` - gradient boosting on all five features (school_code one-hot).

Selection rule (fixed now): highest mean grouped-CV AUC; if within one standard error of the best, prefer the simpler model. Decision threshold: the one maximising balanced accuracy on out-of-fold CV predictions of the chosen model.

In [ ]:
num3 = ['prior_score','session_minutes','hints_used']
def logit(num, cat=()):
    pre = ColumnTransformer([('n', StandardScaler(), list(num)),
                             ('c', OneHotEncoder(handle_unknown='ignore'), list(cat))])
    return make_pipeline(pre, LogisticRegression(max_iter=1000))
def hgb(cat=()):
    pre = ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore', sparse_output=False), list(cat))],
                            remainder='passthrough')
    return make_pipeline(pre, HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0))
all5 = num3 + ['enrol_year','school_code']
candidates = {
 'baseline_prior': (DummyClassifier(strategy='prior'), ['prior_score']),
 'logit_prior':    (logit(['prior_score']), ['prior_score']),
 'logit_session':  (logit(num3), num3),
 'logit_all':      (logit(num3+['enrol_year'], ['school_code']), all5),
 'hgb_session':    (hgb(), num3),
 'hgb_all':        (hgb(['school_code']), all5),
}
complexity = list(candidates)  # listed simplest to most complex

## 2-3. Grouped cross-validation inside the training rows
All preprocessing (scaling, one-hot) lives inside each pipeline, so it is fitted on the training part of each fold only.

In [ ]:
gkf = GroupKFold(n_splits=5)
y, g = train.passed_quiz.values, train.student_key.values
folds = list(gkf.split(train, y, g))
oof = {}; rows = []
for name,(model,cols) in candidates.items():
    p = np.zeros(len(train)); aucs=[]; lls=[]
    for a,b in folds:
        m = clone(model).fit(train.iloc[a][cols], y[a])
        p[b] = m.predict_proba(train.iloc[b][cols])[:,1]
        aucs.append(roc_auc_score(y[b], p[b])); lls.append(log_loss(y[b], p[b]))
    oof[name]=p
    rows.append(dict(model=name, auc=np.mean(aucs), auc_se=np.std(aucs,ddof=1)/np.sqrt(len(aucs)), logloss=np.mean(lls)))
cv = pd.DataFrame(rows).set_index('model'); cv.round(4)

## 5. Choose model and threshold on CV only

In [ ]:
best = cv.auc.idxmax(); thr_ok = cv.auc >= cv.auc[best] - cv.auc_se[best]
chosen = next(n for n in complexity if thr_ok[n])
ths = np.linspace(0.05,0.95,91)
bal = [balanced_accuracy_score(y, oof[chosen]>=t) for t in ths]
threshold = float(ths[int(np.argmax(bal))])
print('best CV AUC:', best, '| chosen (simplest within 1 SE):', chosen, '| threshold:', round(threshold,2),
      '| CV balanced acc at threshold:', round(max(bal),3))
print('baseline CV AUC', round(cv.auc['baseline_prior'],3), '| chosen CV AUC', round(cv.auc[chosen],3))

**Decision (recorded before the test set is scored):** the model, feature set and threshold printed above are final. No changes will be made after seeing test results.

## 6. Refit on all training rows; score the test set once

In [ ]:
model, cols = candidates[chosen]
final = clone(model).fit(train[cols], train.passed_quiz)
pt = final.predict_proba(test[cols])[:,1]; yt = test.passed_quiz.values
base = clone(candidates['baseline_prior'][0]).fit(train[['prior_score']], train.passed_quiz)
pb = base.predict_proba(test[['prior_score']])[:,1]
# bootstrap over students for an interval on the headline AUC
rng = np.random.default_rng(0); ids = test.student_key.unique(); grp = test.groupby('student_key').indices
bs=[]
for _ in range(1000):
    s = rng.choice(ids, len(ids)); ix = np.concatenate([grp[i] for i in s])
    if len(set(yt[ix]))==2: bs.append(roc_auc_score(yt[ix], pt[ix]))
print(f'HEADLINE test AUC ({chosen}): {roc_auc_score(yt,pt):.3f}  95% CI {np.percentile(bs,2.5):.3f}-{np.percentile(bs,97.5):.3f}')
print(f'test log loss {log_loss(yt,pt):.3f} (baseline {log_loss(yt,pb):.3f}); baseline AUC {roc_auc_score(yt,pb):.3f}')
pred = pt>=threshold
print(f'at threshold {threshold:.2f}: accuracy {accuracy_score(yt,pred):.3f}, balanced accuracy {balanced_accuracy_score(yt,pred):.3f}; majority-class accuracy {1-yt.mean():.3f}')

## Conclusion
Only weakly. The chosen model (logistic regression on `prior_score`, `session_minutes`, `hints_used`; threshold 0.41, all picked on grouped CV) scored a held-out test AUC of **0.59** (95% CI 0.57-0.62, bootstrap over students), against 0.50 for the baseline. Test accuracy at the chosen threshold (0.565) is below the majority-class rate (0.582), so the model is not useful as a pass/fail classifier. It is only slightly useful as a ranking or probability estimate (log loss 0.667 vs 0.680).

What carries the signal: `session_minutes` (longer sessions pass more) and `hints_used` (more hints, fewer passes). `prior_score` alone is at chance (CV AUC 0.507). Adding `enrol_year` and `school_code` hurt in CV, and gradient boosting did not beat the logistic model.

Caveats: no time column, so this describes prediction for **new students** from the same population, not for future periods. `session_minutes` and `hints_used` are only usable if prediction happens after the session activity and before the quiz outcome is known. The `waldean` static linter flags three low-confidence 'test-selection' candidates in the selection cell; these are false positives, since that cell uses out-of-fold predictions on training rows only and the test set is scored once, afterwards.